# GB AI — Prototype Deteksi Gempa (Google Colab)

Notebook ini membangun prototype **GB AI** dari nol:
1. Download dataset gempa publik (**ETHZ** via SeisBench, ±22 GB) — data asli dengan label waktu tiba gelombang P.
2. Bikin potongan waveform: **gempa** (P-wave ada di ujung kanan window) dan **noise** (sebelum gempa).
3. Training **Model 1 – Baseline CNN** (deteksi saja).
4. Training **Model 2 – GB AI** (CNN + BiLSTM + attention, deteksi + magnitudo + uncertainty) sesuai Table 1 di paper.
5. Evaluasi: accuracy, precision, recall, F1, false-alarm rate, MAE magnitudo, waktu inference.
6. **Demo sliding window real-time** + ukur waktu deteksi setelah P-wave datang.

### Sebelum mulai
- Menu **Runtime → Change runtime type → T4 GPU → Save**.
- Jalankan cell **berurutan dari atas** (tombol ▶ atau Shift+Enter).
- Hasil potongan data & model disimpan ke **Google Drive**, jadi sesi berikutnya tidak perlu download 22 GB lagi.

> Catatan kejujuran: paper memakai window **60 detik**; prototype ini default **30 detik** karena banyak rekaman ETHZ tidak punya 60 detik data sebelum gempa. Sebutkan ini saat presentasi sebagai penyederhanaan prototype.

## 1. Install library

In [ ]:
!pip install -q seisbench

## 2. Import & pengaturan

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SR = 100                  # sampling rate (Hz)
WIN_SEC = 30              # panjang window (detik). Paper: 60 -> boleh dicoba kalau datanya cukup
WIN = SR * WIN_SEC        # jumlah sampel per window
MAX_TRACES = {"train": 8000, "dev": 1000, "test": 2000}   # jumlah rekaman yang diambil per split
N_DEMO = 200              # jumlah rekaman test untuk demo sliding window
SEED = 42
rng = np.random.default_rng(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "TIDAK ADA - aktifkan T4 GPU di menu Runtime")

## 3. Hubungkan Google Drive (tempat menyimpan hasil)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/GB_AI"
os.makedirs(SAVE_DIR, exist_ok=True)
CACHE = f"{SAVE_DIR}/gbai_windows_{WIN_SEC}s.npz"
print("Folder simpan:", SAVE_DIR)

## 4. Fungsi bantu
- **Bandpass 0.5–20 Hz** (filter kausal, seperti kondisi real-time).
- **Window gempa**: P-wave diletakkan 0.5–10 detik sebelum ujung kanan window — persis kondisi sliding window saat gempa baru datang.
- **Window noise**: berakhir minimal 1 detik sebelum P-wave.
- **Normalisasi z-score** per komponen.

In [ ]:
SOS = butter(4, [0.5, 20], btype="bandpass", fs=SR, output="sos")

def p_arrival(meta):
    # Ambil waktu tiba P paling awal dari semua kolom label P (P, Pg, Pn, P1, ...)
    vals = []
    for k, v in meta.items():
        if k.startswith("trace_P") and k.endswith("_arrival_sample"):
            try:
                if v is not None and not pd.isna(v):
                    vals.append(float(v))
            except (TypeError, ValueError):
                pass
    return int(min(vals)) if vals else None

def normalize(w):
    # w: (3, WIN) -> (WIN, 3)
    w = w - w.mean(axis=1, keepdims=True)
    w = w / (w.std(axis=1, keepdims=True) + 1e-8)
    return w.T.astype(np.float32)

def load_trace(ds, i):
    wf, meta = ds.get_sample(int(i))
    wf = np.nan_to_num(np.asarray(wf, dtype=np.float32))
    if wf.ndim != 2 or wf.shape[0] != 3:
        return None, None, None
    p = p_arrival(meta)
    if p is None or p <= 0 or p >= wf.shape[1]:
        return None, None, None
    wf = sosfilt(SOS, wf, axis=1).astype(np.float32)
    mag = meta.get("source_magnitude", np.nan)
    try:
        mag = float(mag)
    except (TypeError, ValueError):
        mag = np.nan
    return wf, p, mag

MARGIN = 2 * SR   # buang 2 detik awal rekaman (efek awal filter)

def make_windows(ds, n_max):
    X, y, M = [], [], []
    idx = rng.permutation(len(ds))[:n_max]
    for i in tqdm(idx, desc="potong window"):
        wf, p, mag = load_trace(ds, i)
        if wf is None:
            continue
        n = wf.shape[1]
        # window GEMPA: P berada 0.5-10 s sebelum ujung kanan
        end = p + int(rng.uniform(0.5, 10) * SR)
        if end <= n and end - WIN >= MARGIN:
            X.append(normalize(wf[:, end - WIN:end])); y.append(1); M.append(mag)
        # window NOISE: berakhir >= 1 s sebelum P
        last_end = p - SR
        if last_end - WIN >= MARGIN:
            e = int(rng.integers(WIN + MARGIN, last_end + 1))
            X.append(normalize(wf[:, e - WIN:e])); y.append(0); M.append(np.nan)
    return (np.stack(X).astype(np.float16), np.array(y, dtype=np.int8), np.array(M, dtype=np.float32))

PRE = WIN + 10 * SR    # data sebelum P untuk demo
POST = 15 * SR         # data sesudah P untuk demo

def make_demo(ds, n_max):
    segs, mags = [], []
    for i in rng.permutation(len(ds)):
        if len(segs) >= n_max:
            break
        wf, p, mag = load_trace(ds, i)
        if wf is None or p - PRE < MARGIN or p + POST > wf.shape[1]:
            continue
        segs.append(wf[:, p - PRE:p + POST]); mags.append(mag)
    return np.stack(segs).astype(np.float32), np.array(mags, dtype=np.float32)

## 5. Download dataset & potong window
Cell ini **hanya jalan sekali**. Download ETHZ ±22 GB bisa makan waktu 15–40 menit tergantung koneksi Colab.
Kalau file cache sudah ada di Drive, download otomatis dilewati.

In [ ]:
if os.path.exists(CACHE):
    print("Cache ditemukan di Drive -> download dilewati.")
else:
    import seisbench.data as sbd
    data = sbd.ETHZ(sampling_rate=SR, component_order="ZNE")
    print(data)
    train_ds, dev_ds, test_ds = data.train_dev_test()
    print("Rekaman -> train:", len(train_ds), "| dev:", len(dev_ds), "| test:", len(test_ds))

    X_train, y_train, m_train = make_windows(train_ds, MAX_TRACES["train"])
    X_dev, y_dev, m_dev = make_windows(dev_ds, MAX_TRACES["dev"])
    X_test, y_test, m_test = make_windows(test_ds, MAX_TRACES["test"])
    demo_X, demo_m = make_demo(test_ds, N_DEMO)

    np.savez(CACHE, X_train=X_train, y_train=y_train, m_train=m_train,
             X_dev=X_dev, y_dev=y_dev, m_dev=m_dev,
             X_test=X_test, y_test=y_test, m_test=m_test,
             demo_X=demo_X, demo_m=demo_m)
    print("Tersimpan ke", CACHE)

## 6. Load data hasil potongan

In [ ]:
d = np.load(CACHE)
X_train, y_train, m_train = d["X_train"].astype(np.float32), d["y_train"], d["m_train"]
X_dev, y_dev, m_dev = d["X_dev"].astype(np.float32), d["y_dev"], d["m_dev"]
X_test, y_test, m_test = d["X_test"].astype(np.float32), d["y_test"], d["m_test"]
demo_X, demo_m = d["demo_X"], d["demo_m"]

for name, yy in [("train", y_train), ("dev", y_dev), ("test", y_test)]:
    print(f"{name:5s}: {len(yy):6d} window | gempa = {int(yy.sum()):6d} | noise = {int((yy == 0).sum()):6d}")
print("Rekaman demo:", len(demo_X))

# Contoh 1 window gempa & 1 window noise
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
t = np.arange(WIN) / SR
for a, lab in zip(ax, [1, 0]):
    j = int(np.where(y_train == lab)[0][0])
    a.plot(t, X_train[j, :, 0], lw=0.6)
    a.set_title("Contoh window GEMPA (P-wave dekat ujung kanan)" if lab else "Contoh window NOISE")
ax[1].set_xlabel("Detik")
plt.tight_layout(); plt.show()

## 7. Model 1 — Baseline CNN (langkah 1 roadmap)

In [ ]:
def build_baseline():
    inp = keras.Input((WIN, 3))
    x = inp
    for f, k in [(32, 5), (64, 5), (128, 3)]:
        x = layers.Conv1D(f, k, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="detection")(x)
    return keras.Model(inp, out, name="baseline_cnn")

baseline = build_baseline()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
print("Parameter baseline:", f"{baseline.count_params():,}")

hist_b = baseline.fit(X_train, y_train, validation_data=(X_dev, y_dev),
                      epochs=20, batch_size=32,
                      callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)])

## 8. Model 2 — GB AI (sesuai Table 1 paper)
- Conv1D 32 → 64 → 128 (kernel 5, 5, 3; konvolusi pertama stride 2) + BatchNorm + ReLU + MaxPool
- Bidirectional LSTM 256 unit + Dropout 0.5 + additive attention
- 3 output: **deteksi**, **magnitudo**, **uncertainty** (varian magnitudo)
- Loss: L_total = L_detection + 0.5 × L_magnitude + 0.3 × L_uncertainty

Prototype memakai epoch lebih sedikit (maks 30, early stopping patience 5) dibanding rencana paper (150, patience 20) supaya muat di sesi Colab gratis.

In [ ]:
def build_gbai():
    inp = keras.Input((WIN, 3))
    x = inp
    for f, k, s in [(32, 5, 2), (64, 5, 1), (128, 3, 1)]:
        x = layers.Conv1D(f, k, strides=s, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.5)(x)
    # additive attention
    score = layers.Dense(64, activation="tanh")(x)
    score = layers.Dense(1)(score)
    weights = layers.Softmax(axis=1)(score)
    context = layers.Flatten()(layers.Dot(axes=1)([weights, x]))
    det = layers.Dense(1, activation="sigmoid", name="detection")(context)
    mu = layers.Dense(1)(context)
    logvar = layers.Dense(1)(context)
    mag = layers.Concatenate(name="magnitude")([mu, logvar])
    return keras.Model(inp, [det, mag], name="gb_ai")

def mag_loss(y_true, y_pred):
    # y_true = [magnitudo, mask]; noise / magnitudo kosong -> mask 0
    m, mask = y_true[:, 0], y_true[:, 1]
    mu, logvar = y_pred[:, 0], y_pred[:, 1]
    logvar = tf.clip_by_value(logvar, -6.0, 6.0)
    l_mag = tf.abs(m - mu)                                         # L_magnitude
    l_unc = 0.5 * (tf.exp(-logvar) * tf.square(m - mu) + logvar)   # L_uncertainty
    return mask * (0.5 * l_mag + 0.3 * l_unc)

def mag_target(y, m):
    ok = (y == 1) & np.isfinite(m)
    return np.stack([np.where(ok, m, 0.0), ok.astype(np.float32)], axis=1).astype(np.float32)

ym_train, ym_dev, ym_test = mag_target(y_train, m_train), mag_target(y_dev, m_dev), mag_target(y_test, m_test)

gbai = build_gbai()
lr = keras.optimizers.schedules.ExponentialDecay(1e-3, decay_steps=1000, decay_rate=0.9)
gbai.compile(optimizer=keras.optimizers.Adam(lr),
             loss={"detection": "binary_crossentropy", "magnitude": mag_loss},
             metrics={"detection": ["accuracy"]})
gbai.summary()
print("Total parameter GB AI:", f"{gbai.count_params():,}", "(paper memperkirakan ~0.86 juta)")

In [ ]:
hist_g = gbai.fit(X_train, {"detection": y_train, "magnitude": ym_train},
                  validation_data=(X_dev, {"detection": y_dev, "magnitude": ym_dev}),
                  epochs=30, batch_size=32,
                  callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])

plt.figure(figsize=(8, 4))
plt.plot(hist_g.history["loss"], label="train loss")
plt.plot(hist_g.history["val_loss"], label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Kurva training GB AI"); plt.show()

## 9. Evaluasi di data test

In [ ]:
def det_metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0), "F1": f1_score(y_true, pred, zero_division=0),
            "False-alarm rate": fp / max(fp + tn, 1)}

prob_b = baseline.predict(X_test, batch_size=256, verbose=0).ravel()
out_g = gbai.predict(X_test, batch_size=256, verbose=0)
prob_g, mag_pred = out_g[0].ravel(), out_g[1]

results = pd.DataFrame({"Baseline CNN": det_metrics(y_test, prob_b), "GB AI": det_metrics(y_test, prob_g)})
print(results.map(lambda v: f"{v*100:.1f}%"))

ok = (y_test == 1) & np.isfinite(m_test)
mu_test = mag_pred[ok, 0]
sigma_test = np.sqrt(np.exp(np.clip(mag_pred[ok, 1], -6, 6)))
err = np.abs(mu_test - m_test[ok])
print(f"\nMAE magnitudo (semua): {err.mean():.2f}  | rata-rata uncertainty (sigma): {sigma_test.mean():.2f}")
bins = [(-9, 2), (2, 3), (3, 4), (4, 10)]
for lo, hi in bins:
    sel = (m_test[ok] >= lo) & (m_test[ok] < hi)
    if sel.sum() > 0:
        print(f"  M {max(lo,0)}-{hi if hi < 10 else '+'}: MAE = {err[sel].mean():.2f}  (n = {sel.sum()})")

plt.figure(figsize=(5, 5))
plt.scatter(m_test[ok], mu_test, s=4, alpha=0.4)
lim = [np.nanmin(m_test[ok]) - 0.3, np.nanmax(m_test[ok]) + 0.3]
plt.plot(lim, lim, "r--"); plt.xlabel("Magnitudo asli"); plt.ylabel("Prediksi GB AI"); plt.title("Estimasi magnitudo")
plt.show()

## 10. Waktu inference (bukan dikira-kira, tapi diukur)

In [ ]:
def time_model(model, device, n=100):
    x = tf.constant(X_test[:1])
    with tf.device(device):
        model(x, training=False)            # pemanasan
        t0 = time.perf_counter()
        for _ in range(n):
            model(x, training=False)
        return (time.perf_counter() - t0) / n * 1000

timing = {}
for name, mdl in [("Baseline CNN", baseline), ("GB AI", gbai)]:
    timing[name] = {"CPU (ms/window)": time_model(mdl, "/CPU:0")}
    if tf.config.list_physical_devices("GPU"):
        timing[name]["GPU T4 (ms/window)"] = time_model(mdl, "/GPU:0")
print(pd.DataFrame(timing).round(1))
print("\nCatatan: ini diukur di Google Colab, BUKAN di Jetson. Tulis hardware-nya apa adanya saat presentasi.")

## 11. Demo real-time: sliding window
Untuk tiap rekaman demo, window 30 detik digeser setiap **0.1 detik** dari 10 detik sebelum P sampai 15 detik sesudah P.
Alarm berbunyi saat probabilitas gempa ≥ threshold. Hasilnya: **waktu deteksi setelah P-wave** (data latency) dan jumlah **alarm palsu sebelum gempa datang**.

In [ ]:
STEP = int(0.1 * SR)
P_IDX = PRE
THR = 0.5

def sliding_probs(seg, model):
    ends = np.arange(P_IDX - 10 * SR, P_IDX + POST + 1, STEP)
    wins = np.stack([normalize(seg[:, e - WIN:e]) for e in ends])
    p = model.predict(wins, batch_size=256, verbose=0)
    p = p[0] if isinstance(p, list) else p
    return (ends - P_IDX) / SR, p.ravel()

delays, early = [], 0
for seg in tqdm(demo_X, desc="sliding window"):
    tt, pp = sliding_probs(seg, gbai)
    if (pp[tt < 0] >= THR).any():
        early += 1                       # alarm sebelum P = alarm palsu
    hit = np.where((tt >= 0) & (pp >= THR))[0]
    delays.append(tt[hit[0]] if len(hit) else np.nan)

delays = np.array(delays)
det = np.isfinite(delays)
print(f"Rekaman demo                 : {len(delays)}")
print(f"Terdeteksi (<= 15 s setelah P): {det.mean()*100:.1f}%")
print(f"Median waktu deteksi         : {np.nanmedian(delays):.2f} s setelah P-wave")
print(f"Rata-rata waktu deteksi      : {np.nanmean(delays):.2f} s")
print(f"Terdeteksi dalam <= 3 s      : {(delays <= 3).mean()*100:.1f}%")
print(f"Rekaman dengan alarm sebelum P (false trigger): {early} ({early/len(delays)*100:.1f}%)")

plt.figure(figsize=(7, 4))
plt.hist(delays[det], bins=30)
plt.axvline(3, color="r", ls="--", label="Target paper: 3 s")
plt.xlabel("Detik setelah P-wave"); plt.ylabel("Jumlah rekaman"); plt.title("Waktu deteksi GB AI"); plt.legend(); plt.show()

In [ ]:
# Visualisasi 1 contoh (ganti angka k untuk lihat rekaman lain)
k = 0
seg = demo_X[k]
tt, pp = sliding_probs(seg, gbai)
t_full = (np.arange(seg.shape[1]) - P_IDX) / SR

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(t_full, seg[0], lw=0.5, color="k")
ax[0].axvline(0, color="b", ls="--", label="P-wave tiba")
ax[0].set_ylabel("Komponen Z"); ax[0].legend()
ax[1].plot(tt, pp, color="tab:orange", label="Probabilitas gempa")
ax[1].axhline(THR, color="gray", ls=":")
alert = np.where((tt >= 0) & (pp >= THR))[0]
if len(alert):
    ax[1].axvline(tt[alert[0]], color="r", label=f"ALERT di +{tt[alert[0]]:.1f} s")
    ax[0].axvline(tt[alert[0]], color="r")
ax[1].set_xlim(-10, POST / SR); ax[1].set_ylim(-0.05, 1.05)
ax[1].set_xlabel("Detik relatif terhadap P-wave"); ax[1].legend()
plt.suptitle(f"GB AI sliding-window demo — magnitudo katalog: {demo_m[k]:.1f}")
plt.tight_layout(); plt.show()

## 12. Simpan model & ringkasan hasil (untuk PPT)

In [ ]:
gbai.save(f"{SAVE_DIR}/gb_ai.keras")
baseline.save(f"{SAVE_DIR}/baseline_cnn.keras")

summary = {
    "Dataset": "ETHZ (SeisBench), resample 100 Hz, window %d s" % WIN_SEC,
    "Window test (gempa/noise)": f"{int(y_test.sum())}/{int((y_test == 0).sum())}",
    "Parameter GB AI": f"{gbai.count_params():,}",
    "GB AI accuracy": f"{results.loc['Accuracy', 'GB AI']*100:.1f}%",
    "GB AI F1": f"{results.loc['F1', 'GB AI']*100:.1f}%",
    "GB AI false-alarm rate": f"{results.loc['False-alarm rate', 'GB AI']*100:.1f}%",
    "MAE magnitudo": f"{err.mean():.2f}",
    "Median deteksi setelah P": f"{np.nanmedian(delays):.2f} s",
    "Inference GB AI (CPU)": f"{timing['GB AI']['CPU (ms/window)']:.1f} ms",
}
for k_, v_ in summary.items():
    print(f"{k_:28s}: {v_}")
pd.Series(summary).to_csv(f"{SAVE_DIR}/hasil_prototype.csv")
print("\nModel & ringkasan tersimpan di", SAVE_DIR)
print("Untuk load ulang: keras.models.load_model(path, custom_objects={'mag_loss': mag_loss})")